In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.4 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file

In [3]:
# Setup observability
USE_OBSERVABILITY = True

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )

In [4]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient

In [5]:
from massi.runtimes.jupyter.ui import JupyterTextWidgetIOPromptClient

In [6]:
from massi.agents.massi import create_ollama_llm

base_config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
base_deps = Dependencies(llm_client=create_ollama_llm(base_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [7]:
print("... using standard dependencies ...")
factory = MassiScenarioFactory(base_config)
graph, _ = factory.resource_properties_graph(base_deps)

... using standard dependencies ...


In [8]:
import grandalf

print(graph.get_graph().draw_mermaid())
graph.get_graph().print_ascii()

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate_resource_metadata(validate_resource_metadata)
	interpret_metadata_checks(interpret_metadata_checks)
	plan_resource_interview(plan_resource_interview)
	pull_next_resource_query(pull_next_resource_query)
	execute_resource_query(execute_resource_query)
	register_resource_response(register_resource_response)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate_resource_metadata;
	execute_resource_query --> register_resource_response;
	interpret_metadata_checks --> plan_resource_interview;
	plan_resource_interview --> pull_next_resource_query;
	pull_next_resource_query --> execute_resource_query;
	register_resource_response -. &nbsp;QueryPlanExecutionState.STOP&nbsp; .-> __end__;
	register_resource_response -. &nbsp;QueryPlanExecutionState.ACT&nbsp; .-> pull_next_resource_query;
	validate_resource_metadata --> interpret_metadata_checks;
	classDef default fill:#f2f0ff,line-height:1.

In [9]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [10]:
import pprint

In [11]:
from massi.runtimes.flow import stream_until_iterruption

In [12]:
from unittest.mock import patch

MOCK_RUN = False

In [13]:
from massi.agents.constructs import GroupedGisResourceFile

graph_invocation_input_state = {
    "resource":  GroupedGisResourceFile(
        file_location='data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', 
        filename='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', 
        extension='.csv', 
        author='jovyan', 
        gis_relevancy_score=1.0, 
        resource_name='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3', 
        group_score=1.0,
    )
}

In [14]:
stream_until_iterruption(graph, graph_invocation_input_state, config=thread, vebose=True)

{'messages': [], 'resource': GroupedGisResourceFile(file_location='data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', filename='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', extension='.csv', author='jovyan', gis_relevancy_score=1.0, resource_name='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3', group_score=1.0)}


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Parsing LLM output produced both a final answer and a parse-able action:: **Thought:** To determine the schema that the data follows, we first need to inspect the DataFrame loaded from the CSV file (`Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv`). This involves checking the column names, data types, and any spatial or coordinate-related attributes such as CRS (Coordinate Reference System) or bounding box. We also compare the observed metadata with the JSON schema provided (which includes keys like `location`, `type`, `format`, `schema`, `sources`, `description`, `creator`, `geometry_name`, `bounding_box`, `crs`, and `update_frequency`). 

**Action:** python_repl_ast
**Action Input:** print(df.head())

**Observation:** 
|          |   FID |   OBJECTID | WWD_NO                      | PWSID        

In [15]:
graph.get_state(thread).values["query_plan"]

Deserializing unregistered type massi.agents.constructs.GroupedGisResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'GroupedGisResourceFile')]


{'queries': [{'id': 1,
   'key': 'location',
   'json_path': '$.location',
   'priority': 5,
   'query': 'Please provide the primary location where the digital geospatial data is stored or accessed. This could be a file path, a URI, or a server location. What is the exact location where this data resides?'},
  {'id': 2,
   'key': 'type',
   'json_path': '$.type',
   'priority': 4,
   'query': 'What is the type or format of the digital geospatial data? For example, is it a GIS file like a shapefile, GeoJSON, or a raster image?'},
  {'id': 3,
   'key': 'format',
   'json_path': '$.format',
   'priority': 4,
   'query': 'Please specify the format of the data. For instance, is it in GeoJSON, Shapefile, or another specific GIS format?'},
  {'id': 4,
   'key': 'schema',
   'json_path': '$.schema',
   'priority': 3,
   'query': 'What is the schema or standard that this data follows? For example, is it compliant with a specific GIS standard like GeoPackage or a specific coordinate reference sy

In [16]:
graph.get_state(thread).values["current_query_step"]

Deserializing unregistered type massi.agents.constructs.GroupedGisResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'GroupedGisResourceFile')]


{'id': 4,
 'key': 'schema',
 'json_path': '$.schema',
 'priority': 3,
 'query': 'What is the schema or standard that this data follows? For example, is it compliant with a specific GIS standard like GeoPackage or a specific coordinate reference system?'}

In [17]:
graph.get_state(thread).values["current_query_response"]

Deserializing unregistered type massi.agents.constructs.GroupedGisResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'GroupedGisResourceFile')]


'CSV\n```'

In [18]:
graph.get_state(thread).values["updated_metadata"]

Deserializing unregistered type massi.agents.constructs.GroupedGisResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'GroupedGisResourceFile')]


{'metadata': {'location': 'data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv',
  'type': "The provided CSV preview does not specify the exact type or format of the digital geospatial data. The presence of 'Shapearea' and 'Shapelen' columns suggests that the data includes geographic information, but without additional context, the format cannot be definitively identified.",
  'format': 'CSV',
  'schema': {},
  'sources': [],
  'description': 'this is my response',
  'creator': [],
  'geography_name': '',
  'bbox': [],
  'crs': '',
  'update_frequency': 'as_needed'}}